# IOAI — 2024 First Stage Imbalanced Classification (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/test.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-imbalanced-classification/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 불균형 분류 — 모범답안 (동결 ResNet18 + 선형 헤드)

ImageNet 사전학습 **ResNet18** 을 특징추출기로 동결하고 **선형 헤드**만 클래스 가중치로 학습한다. 소량·불균형 데이터에서도 강한 사전학습 특징 덕분에 균형 테스트 정확도가 높다(≈0.95+).

## 데이터 로드(+클래스 가중치)

In [ ]:
import glob, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)

tf = transforms.Compose([transforms.Resize((64,64)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class DS(Dataset):
    def __init__(self, files, labels=None): self.files, self.labels = files, labels
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        x = tf(Image.open(self.files[i]).convert("RGB"))
        return x, (-1 if self.labels is None else self.labels[i])
tr = pd.read_csv("data/train.csv")
train_files = ["data/train/"+f for f in tr["filename"]]; train_labels = tr["label"].tolist()
test_df = pd.read_csv("data/test.csv"); test_files = ["data/test/"+f for f in test_df["filename"]]
test_ids = [f.rsplit("/",1)[1] for f in test_files]
print("train", len(train_files), "(onion", sum(train_labels), ") test", len(test_files))
# 불균형 대응 클래스 가중치
w = torch.tensor([1/ (train_labels.count(0)), 1/(train_labels.count(1))], dtype=torch.float32)
w = (w/w.sum()*2).to(device)
train_dl = DataLoader(DS(train_files, train_labels), batch_size=64, shuffle=True)
test_dl  = DataLoader(DS(test_files), batch_size=64)

## 동결 ResNet18 + 선형 헤드

In [ ]:
from torchvision import models
m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in m.parameters(): p.requires_grad = False          # 백본 동결
m.fc = nn.Linear(m.fc.in_features, 2)                       # 선형 헤드만 학습
m = m.to(device)

## 학습

In [ ]:
crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], 1e-3)
for e in range(12):
    m.train(); tot=0
    for x,y in train_dl:
        x,y=x.to(device),y.to(device); opt.zero_grad(); loss=crit(m(x),y); loss.backward(); opt.step(); tot+=loss.item()*len(x)
    print(f"epoch {e} loss {tot/len(train_files):.4f}")

## 예측 → submission.csv

In [ ]:
m.eval(); preds=[]
with torch.no_grad():
    for x,_ in test_dl: preds += m(x.to(device)).argmax(1).cpu().tolist()
pd.DataFrame({"id": test_ids, "label": preds}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(preds), "| onion predicted", sum(preds))

백본 일부 미세조정·증강을 더하면 더 오를 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)